# 子代理模式

- 管理 完全由Agent主导 跟工作流没太大关系了，就是工具调用。
    - tool (通过工具调用代理)
        - Agent

In [1]:
import os
from typing import List, Dict, Any

from langchain.agents import create_agent
from langchain.tools import tool
from langchain.chat_models import init_chat_model

## 构建子代理

### 工具调

In [16]:
@tool
def execute_python_code(code: str) -> str:
    """
    执行Python代码并返回结果。
    用于测试代码片段、计算或简单脚本执行。
    
    Args:
        code: 要执行的Python代码字符串
    """
    print(f"[exec]要执行的代码{code}")
    try:
        # 安全限制：只允许有限的builtins
        safe_globals = {
            # "__builtins__": {
            #     "print": print,
            #     "len": len,
            #     "range": range,
            #     "int": int,
            #     "float": float,
            #     "str": str,
            #     "list": list,
            #     "dict": dict,
            #     "sum": sum,
            #     "abs": abs,
            # }
        }
        exec_globals = {}
        exec(code, safe_globals, exec_globals)
        
        # 尝试获取执行结果
        if "result" in exec_globals:
            output = str(exec_globals["result"])
        else:
            output = "代码执行成功（无返回值）"
        
        return f"执行结果:\n{output}"
    except Exception as e:
        return f"执行错误: {str(e)}"

### 构建代理

In [17]:
def create_code_agent():
    """创建代码助手子Agent"""
    return create_agent(
        model="ollama:qwen2.5:7b-instruct-q4_K_M",
        tools=[execute_python_code],
        system_prompt="""你是一个专业的代码助手，擅长编写和执行Python代码。
        
        你的职责：
        1. 根据用户需求编写Python代码
        2. 执行代码并解释结果
        3. 调试代码错误
        
        重要提示：
        - 使用 execute_python_code 工具执行代码
        - 对于复杂问题，先拆解为多个小步骤
        - 返回结果时要清晰、易懂
        """
    )

## 代理

### 工具

In [18]:
@tool
def web_search(query: str, max_results: int = 3) -> str:
    """
    模拟网络搜索工具。
    在实际生产环境中，可以接入 Tavily、SerpAPI 等搜索服务。
    
    Args:
        query: 搜索查询词
        max_results: 最大返回结果数
    """
    print(f"[search]查询{query}")
    # 这是一个模拟实现，实际应接入真实的搜索API
    # 例如：tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
    mock_results = {
        "LangChain": "LangChain 是一个用于构建 LLM 应用的开源框架。最新版本 v1.0 专注于 Agent 开发。",
        "Python": "Python 是广泛使用的编程语言，以简洁易读著称。",
        "AI": "人工智能正在快速发展，LLM 是当前热点方向。"
    }
    
    for key, value in mock_results.items():
        if key.lower() in query.lower():
            return f"搜索结果:\n- {key}: {value}"
    
    return f"未找到关于 '{query}' 的详细信息。建议使用更具体的关键词。"

### 构建代理

In [19]:
def create_research_agent():
    """创建研究助手子Agent"""
    return create_agent(
        model="ollama:qwen2.5:7b-instruct-q4_K_M",
        tools=[web_search],
        system_prompt="""你是一个专业的研究助手，擅长信息搜索和知识整理。
        
        你的职责：
        1. 搜索用户需要的信息
        2. 整合搜索结果，提供总结
        3. 引用信息来源
        
        重要提示：
        - 使用 web_search 工具获取信息
        - 搜索结果可能不完整，如有需要可进行多次搜索
        - 用结构化的方式呈现信息
        """
    )

## 管理者

### 使用子代理构建工具 (托管)

In [20]:
@tool
def delegate_to_code_agent(task: str) -> str:
    """
    将任务委派给代码助手子Agent。
    代码助手擅长编写和执行Python代码、解决编程问题。
    
    Args:
        task: 需要代码助手处理的任务描述
    """
    print(f">>>>>>执行code任务:{task}中")
    code_agent = create_code_agent()
    result = code_agent.invoke({
        "messages": [{"role": "user", "content": task}]
    })
    return f"[代码助手回复]\n{result['messages'][-1].content}"

### 使用子代理构建工具 (托管)

In [21]:
@tool
def delegate_to_research_agent(task: str) -> str:
    """
    将任务委派给研究助手子Agent。
    研究助手擅长信息搜索、知识整理和资料汇总。
    
    Args:
        task: 需要研究助手处理的任务描述
    """
    print(f">>>>>>执行research任务:{task}中")
    research_agent = create_research_agent()
    result = research_agent.invoke({
        "messages": [{"role": "user", "content": task}]
    })
    return f"[研究助手回复]\n{result['messages'][-1].content}"

### 构建管理者代理

In [22]:
def create_supervisor_agent():
    """创建主管Agent，负责调度子Agent"""
    
    supervisor_prompt = """你是一个智能任务调度主管。你的职责是将复杂任务拆解并分配给最合适的专家助手。

    ## 可用的专家助手
    
    1. **代码助手** (通过 delegate_to_code_agent 调用)
       - 擅长：编写代码、执行代码、调试程序、数据计算
       - 示例任务："写一个计算斐波那契数列的函数"
    
    2. **研究助手** (通过 delegate_to_research_agent 调用)
       - 擅长：搜索信息、知识整理、资料汇总
       - 示例任务："介绍一下LangChain的主要功能"
    
    ## 任务分配原则
    
    - 编程相关问题 → 委派给代码助手
    - 信息查询/研究 → 委派给研究助手
    - 复合任务（如"研究并实现"）→ 拆解后依次委派
    - 简单问题可以直接回答，无需委派
    
    ## 调度流程
    
    1. 分析用户请求，判断需要哪些专家
    2. 如需要多个专家，依次调用相应的委派工具
    3. 整合各专家的结果，返回完整答案
    
    记住：每个委派工具调用时，要提供清晰、完整的任务描述。
    """
    
    return create_agent(
        model="ollama:qwen2.5:7b-instruct-q4_K_M",
        tools=[delegate_to_code_agent, delegate_to_research_agent],
        system_prompt=supervisor_prompt
    )

## 通过管理者代理完成任务

In [26]:
supervisor = create_supervisor_agent()
messages = {
    "messages": [{"role": "user", "content": "Python？"},
                {"role": "user", "content": "生成一个冒泡算法，并用数据[22, 2, 8, 6, 19]测试执行下。"}]
}
result = supervisor.invoke(messages)
print(result['messages'][-1].content)

>>>>>>执行code任务:编写一个冒泡排序算法，并用数据 [22, 2, 8, 6, 19] 进行测试。中
[exec]要执行的代码def bubble_sort(arr):
    n = len(arr)
    for i in range(n):
        # 标志位用于检测这一轮是否有元素交换
        swapped = False
        for j in range(0, n - i - 1):
            if arr[j] > arr[j + 1]:
                # 交换元素
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
                swapped = True
        # 如果没有元素交换，说明数组已经有序，提前终止
        if not swapped:
            break
    return arr

# 测试数据
data = [22, 2, 8, 6, 19]
bubble_sort(data)
print(data)
[2, 6, 8, 19, 22]
代码助手已经为您编写了冒泡排序算法，并使用数据 `[22, 2, 8, 6, 19]` 进行了测试。排序后的结果是 `[2, 6, 8, 19, 22]`。这里是有助于您理解的代码：

```python
def bubble_sort(arr):
    n = len(arr)
    for i in range(n):
        # 标志位用于优化，如果某一轮没有发生交换，说明数组已经有序
        swapped = False
        for j in range(0, n-i-1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
                swapped = True
        # 如果没有发生交换，提前退出
        if not swapped:
            break

# 测试数据
test_data 